# 🛡️ Kaggle 24/7 OSINT Agent + Tunnel Cloudflare & Keep-Alive
Plateforme OSINT 24/7 alimentée par Qwen3.6-12B GGUF. Boucle infinie active pour empêcher l'arrêt du container.

In [ ]:
# 1. Redirection TMPDIR & Initialisation des dossiers
import os

os.environ['TMPDIR'] = '/kaggle/working/tmp'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/tmp/pip'
os.makedirs('/kaggle/working/tmp', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)
print('🟢 Dossiers temporaires et modèles prêts !')

In [ ]:
# 2. Clonage ou Mise à jour par Git Pull
import os, subprocess

repo_dir = '/kaggle/working/projet_osint'
clone_url = 'https://github.com/whbky6vqjb-coder/osint.git'

if os.path.exists(repo_dir):
    print('⚡ Dépôt déjà présent : Exécution d\'un Git Pull (1 sec)...')
    subprocess.run(['git', '-C', repo_dir, 'pull', 'origin', 'main'])
    print('🟢 Code source mis à jour !')
else:
    print('📥 Premier clonage du dépôt Git...')
    subprocess.run(['git', 'clone', clone_url, repo_dir])
    print('🟢 Dépôt Git cloné !')

!pip install --no-cache-dir --prefer-binary huggingface_hub "llama-cpp-python[server]"

In [ ]:
# 4. Tunnel HTTPS Cloudflare redirigé vers llama-server (Port 8080) & Envoi de l'URL à Render
import os, sys, subprocess, time, re, urllib.request, json

print('Installation de cloudflared...')
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared || true

subprocess.run(['pkill', '-f', 'cloudflared'])
time.sleep(1)

print('Lancement du Tunnel HTTPS Cloudflare pour llama-server (Port 8080)...')
tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8080'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url_found = None
render_url = os.environ.get('RENDER_URL', 'https://osint-app.onrender.com')
llm_secret = os.environ.get('LLM_URL_SECRET', 'default_secret')

for _ in range(40):
    line = tunnel_process.stdout.readline()
    if line and 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url_found = match.group(0)
            print('\n======================================================')
            print(f'🚀 LLM TUNNEL EST EN LIGNE : {url_found}')
            print('======================================================\n')
            
            # POST au serveur sur Render
            print(f'Envoi de la nouvelle URL LLM au serveur Render ({render_url})...')
            try:
                req = urllib.request.Request(
                    f'{render_url}/api/internal/update-llm-url',
                    data=json.dumps({'url': f'{url_found}/v1'}).encode('utf-8'),
                    headers={
                        'Content-Type': 'application/json',
                        'X-Secret': llm_secret
                    },
                    method='POST'
                )
                with urllib.request.urlopen(req) as response:
                    res_data = json.loads(response.read().decode())
                    print(f'✅ Serveur Render mis à jour : {res_data}')
            except Exception as e:
                print(f'❌ Échec de mise à jour Render : {e}')
            break
    time.sleep(0.5)

if not url_found:
    print('⚠️ Capture Cloudflare : Le tunnel s\'initialise en tâche de fond.')

In [ ]:
# 4. Démarrage de FastAPI & Tunnel HTTPS Cloudflare (Capture d'URL Ultra-Fiable)
import os, sys, subprocess, time, re

backend_dir = '/kaggle/working/projet_osint/backend'
os.chdir(backend_dir)
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

!pip install --no-cache-dir -r requirements.txt
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared || true

subprocess.run(['pkill', '-f', 'app.main'])
time.sleep(1)

print('Démarrage du Serveur FastAPI (Port 8000)...')
server_process = subprocess.Popen(['python', '-m', 'app.main'], cwd=backend_dir)
time.sleep(5)

subprocess.run(['pkill', '-f', 'cloudflared'])
time.sleep(1)

print('Lancement du Tunnel HTTPS...')
tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url_found = None
start_time = time.time()
for _ in range(40):
    line = tunnel_process.stdout.readline()
    if line and 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url_found = match.group(0)
            with open('/kaggle/working/online_url.txt', 'w') as f:
                f.write(url_found)
            print('\n======================================================')
            print(f'🚀 VOTRE INTERFACE OSINT EST EN LIGNE : {url_found}')
            print('======================================================\n')
            break
    time.sleep(0.5)

if not url_found:
    print('⚠️ Capture Cloudflare : Le tunnel s\'initialise en tâche de fond.')

In [ ]:
# 5. Boucle d'exécution continue 24/7 (Infinie) - Empêche la fermeture de Kaggle
import time

print('🟢 Serveur actif 24/7. Boucle d\'écoute en cours...')
counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 30 == 0:
        print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")} 🟢] Le serveur OSINT tourne depuis {counter} minutes.')